Stage 1: Mapping all category to each table in URT

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../raw/urt_raw.csv")

In [2]:
# ============================================================
# 1. Initial category by row block
# ============================================================

# Standardize column names
df.columns = df.columns.str.strip()

# Optional: rename to cleaner column names
df = df.rename(columns={
    "Nama Pangan": "food_name",
    "Berat dalam Gram": "weight_gram",
    "Ukuran Rumah Tangga (URT)": "urt"
})

# Clean text columns
df["food_name"] = df["food_name"].astype(str).str.strip()
df["urt"] = df["urt"].astype(str).str.strip()

category_blocks = [
    (0, 29, "MP", "Makanan Pokok"),
    (30, 42, "LN", "Lauk Nabati"),
    (43, 107, "LH", "Lauk Hewani"),
    (108, 148, "B", "Buah"),
    (144, 150, "G", "Gula"),
    (151, 165, "M", "Minyak / Lemak"),
]

df["category_code"] = np.nan
df["category_name"] = np.nan

for start, end, code, name in category_blocks:
    df.loc[start:end, "category_code"] = code
    df.loc[start:end, "category_name"] = name


# ============================================================
# 2. Correction rules
# ============================================================

susu_names = [
    "Susu sapi",
    "Susu kerbau",
    "Susu kambing"
]

df.loc[df["food_name"].isin(susu_names), "category_code"] = "SS"
df.loc[df["food_name"].isin(susu_names), "category_name"] = "Susu"

C:\Users\azraf\AppData\Local\Temp\ipykernel_12924\3200040109.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'MP' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[start:end, "category_code"] = code
C:\Users\azraf\AppData\Local\Temp\ipykernel_12924\3200040109.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Makanan Pokok' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[start:end, "category_name"] = name


In [3]:
# the data already include all 7 types just missing the S (Sayur) due to the urt doesn't provide any sayur, so we can add sayur because sayur additional in 100 gram on 1 gelas
print(df["category_code"].value_counts(dropna=False))


category_code
LH    62
B     36
MP    30
LN    13
M     13
G      7
SS     3
Name: count, dtype: int64


In [4]:
import pandas as pd
import re
import unicodedata

def basic_clean_name(name):
    """
    Basic cleaning:
    - lowercase
    - remove extra spaces
    - normalize unicode
    - standardize punctuation spacing
    """
    if pd.isna(name):
        return ""

    name = str(name)
    name = unicodedata.normalize("NFKC", name)
    name = name.lower().strip()
    name = re.sub(r"\s+", " ", name)
    name = name.replace(" ,", ",")
    name = name.replace(", ", ", ")
    return name


def normalize_tkpi_name_for_matching(name):
    """
    Conservative normalization for matching TKPI food names to URT names.
    This removes general preparation descriptors but keeps identity-changing words.
    """
    name = basic_clean_name(name)

    # Remove common descriptors that usually describe physical state/preparation.
    # Be conservative: do NOT remove 'asin', 'asap', 'dendeng', 'sarden', etc.
    removable_phrases = [
        ", segar",
        ", mentah",
        ", rebus",
        ", kukus",
        ", masakan",
        ", matang",
        ", bakar",
    ]

    for phrase in removable_phrases:
        name = name.replace(phrase, "")

    # Some words are safe to remove only if they appear at the end
    # or after comma as descriptors.
    name = re.sub(r",\s*segar$", "", name)
    name = re.sub(r",\s*mentah$", "", name)
    name = re.sub(r",\s*rebus$", "", name)
    name = re.sub(r",\s*kukus$", "", name)
    name = re.sub(r",\s*masakan$", "", name)
    name = re.sub(r",\s*matang$", "", name)

    # Normalize separators
    name = name.replace("/", " ")
    name = re.sub(r"\s+", " ", name)
    name = name.strip(" ,.-")

    return name



In [5]:
df["urt_food_name_clean"] = df["food_name"].apply(basic_clean_name)
df["urt_food_name_match"] = df["food_name"].apply(normalize_tkpi_name_for_matching)

In [6]:
output_file = "urt_with_category.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig")